In [9]:
from datasets import load_dataset
dataset=load_dataset("opus_books","en-es")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

en-es/train-00000-of-00001.parquet:   0%|          | 0.00/16.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/93470 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 93470
    })
})


In [10]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
from transformers import MarianTokenizer, MarianMTModel
model_name="Helsinki-NLP/opus-mt-es-en"
tokenizer=MarianTokenizer.from_pretrained(model_name)
model=MarianMTModel.from_pretrained(model_name)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [12]:
max_length=128
def preprocessing(example):
    inputs = example["translation"]["es"]
    targets = example["translation"]["en"]

    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [13]:
tokenized_dataset=dataset.map(preprocessing)

Map:   0%|          | 0/93470 [00:00<?, ? examples/s]

In [14]:
from transformers import TrainingArguments,Trainer
training_args=TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=5,
    logging_steps=100,
    save_steps=500
)

In [15]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].select(range(2000)),
)
trainer.train()

Step,Training Loss
100,1.551306
200,0.959877
300,0.761715
400,0.645838
500,0.675393
600,0.527270
700,0.532780
800,0.469141
900,0.424366
1000,0.448376


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.6390614074707032, metrics={'train_runtime': 336.7763, 'train_samples_per_second': 29.693, 'train_steps_per_second': 3.712, 'total_flos': 338983649280000.0, 'train_loss': 0.6390614074707032, 'epoch': 5.0})

In [16]:
def translate(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(**inputs)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(translate("Hola"))

Hiya.
